 ## ----------------- DOWNLOADING NEUROSYNTH -----------------

In [ ]:
import os
from pprint import pprint

from nimare.extract import download_abstracts, fetch_neuroquery, fetch_neurosynth
from nimare.io import convert_neurosynth_to_dataset

# biopython is unnecessary here, but is required by download_abstracts.
# We import it here only to document the dependency and cause an early failure if it's missing.
import Bio  # pip install biopython

out_dir = os.path.abspath("../example_data/") # Set your pathway
os.makedirs(out_dir, exist_ok=True)

files = fetch_neurosynth(
    data_dir=out_dir,
    version="7",
    overwrite=False,
    source="abstract",
    vocab="terms",
)
# Note that the files are saved to a new folder within "out_dir" named "neurosynth".
pprint(files)
neurosynth_db = files[0]

In [ ]:
neurosynth_dset = convert_neurosynth_to_dataset(
    coordinates_file=neurosynth_db["coordinates"],
    metadata_file=neurosynth_db["metadata"],
    annotations_files=neurosynth_db["features"],
)
neurosynth_dset.save(os.path.join(out_dir, "neurosynth_dataset.pkl.gz"))
print(neurosynth_dset)

In [ ]:
neurosynth_dset = download_abstracts(neurosynth_dset, "user@your_university") # your email
neurosynth_dset.save(os.path.join(out_dir, "neurosynth_dataset_with_abstracts.pkl.gz"))

In [15]:
import gzip
import pickle

# Open and decompress the .pkl.gz file
with gzip.open('neurosynth_dataset_with_abstracts.pkl.gz', 'rb') as f:
    neurosynth_dset = pickle.load(f)


TRAINING DECODER

In [ ]:
from nimare.decode.continuous import CorrelationDecoder
from nimare.meta.cbma import mkda

decoder = CorrelationDecoder(
    frequency_threshold=0.001,  
    meta_estimator=mkda.MKDAChi2, 
)
decoder.fit(neurosynth_dset) 

## DECODE!

In [ ]:
# WE already have the decoder model ready... import! or if you just download and trained the decoder, better to save it!
import pickle

with open('decoder.pkl', 'rb') as f:
    decoder = pickle.load(f)

In [ ]:
# now let's decode each state and output result as excel
for i in range(1, 11):
    file_name = f'State_activation/state{i}.nii.gz'
    print(f'-------- Processing STATE {i} -----------')
    
    decoding_results = decoder.transform(file_name)
    
    output_csv = f'neurosynth_output/State{i}.csv'
    
    decoding_results.to_csv(output_csv, index_label='feature')

## Graph Neurosynth


In [7]:
import plotly.graph_objects as go
import pandas as pd

labels = ['monetary reward','moral','memory','emotion','flexibility','sensorimotor',
          'mentalizing','self referential','conflict','memory retrieval','negative emotional',
          'food','attentional control','visual information','cognitive control','reward anticipation']

matlab_colors = [
    [166, 206, 227],   # Light Blue
    [31, 120, 180],    # Dark Blue
    [178, 223, 138],   # Light Green
    [51, 160, 44],     # Dark Green
    [251, 154, 153],   # Light Red
    [227, 26, 28],     # Dark Red
    [253, 191, 111],   # Light Orange
    [255, 127, 0],     # Orange
    [202, 178, 214],   # Light Purple
    [106, 61, 154]     # Dark Purple
]

colors = [f'rgb({r},{g},{b})' for r, g, b in matlab_colors]

data = {}

for i in range(1, 11):  
    file_name = f'neurosynth_output/State{i}.csv'
    state_data = pd.read_csv(file_name)
    
    values = []
    for label in labels:
        term = f'terms_abstract_tfidf__{label}'
        matching_row = state_data[state_data['feature'] == term]
        
        if not matching_row.empty:
            values.append(matching_row['r'].values[0])
        else:
            values.append(float('nan'))
    
    data[f'State{i}'] = values

In [8]:
labels = ['mon reward','moral','mem','emo','flex','sensmot',
          'ment','self ref','conf','mem retr','neg emo',
          'food','atten cont','vis info','cog cont','rew anti']

fig = go.Figure()
    
fig.add_trace(go.Scatterpolar(
        r=data['State1'],  
        theta=labels,  
        fill='toself',  
        name='state1',  
        line=dict(color=colors[0])  
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            tickvals=[-0.2, 0, 0.2],
            range=[-0.2, 0.3],
            tickangle=0 
        )),
    showlegend=True,
    title=f"State 1",
    font=dict(
        family="Helvetica", 
        size=20,
        color="black",
    )
)
fig.show()

In [9]:
fig = go.Figure()
    
fig.add_trace(go.Scatterpolar(
        r=data['State2'],  
        theta=labels,  
        fill='toself',  
        name='state2',  
        line=dict(color=colors[1])  
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            tickvals=[ -0.1, 0, 0.1],
            range=[-0.15, 0.1],
            tickangle=0 
        )),
    showlegend=True,
    title=f"State 2",
        font=dict(
        family="Helvetica", 
        size=20,
        color="black",
    )
        )
fig.show()

In [10]:

fig = go.Figure()
    
fig.add_trace(go.Scatterpolar(
        r=data['State3'],  
        theta=labels,  
        fill='toself',  
        name='state3',  
        line=dict(color=colors[2])  
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            tickvals=[0, 0.15, 0.3],
            range=[-0.2, 0.34],
            tickangle=0 
        )),
    showlegend=True,
    title=f"State 3",
        font=dict(
        family="Helvetica", 
        size=20,
        color="black",
    )
        )
fig.show()

In [11]:

fig = go.Figure()
    
fig.add_trace(go.Scatterpolar(
        r=data['State4'],  
        theta=labels,  
        fill='toself',  
        name='state4',  
        line=dict(color=colors[3])  
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            tickvals=[-0.2, 0, 0.1],
            range=[-0.2, 0.15],  
            tickangle=0
        )),
    showlegend=True,
    title=f"State 4",
        font=dict(
        family="Helvetica", 
        size=20,
        color="black",
    )
        )
fig.show()

In [12]:

fig = go.Figure()
    
fig.add_trace(go.Scatterpolar(
        r=data['State5'],  
        theta=labels,  
        fill='toself',  
        name='state5',  
        line=dict(color=colors[4])  
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            tickvals=[-0.1,0, 0.1,0.2],
            tickangle=0
        )),
    showlegend=True,
    title=f"State 5",
        font=dict(
        family="Helvetica", 
        size=20,
        color="black",
    )
        )
fig.show()

In [13]:

fig = go.Figure()
    
fig.add_trace(go.Scatterpolar(
        r=data['State6'],  
        theta=labels,  
        fill='toself',  
        name='state6',  
        line=dict(color=colors[5])  
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            tickvals=[-0.15,0,0.1],
            tickangle=0
        )),
    showlegend=True,
    title=f"State 6",
        font=dict(
        family="Helvetica", 
        size=20,
        color="black",
    )
        )
fig.show()

In [14]:

fig = go.Figure()
    
fig.add_trace(go.Scatterpolar(
        r=data['State7'],  
        theta=labels,  
        fill='toself',  
        name='state7',  
        line=dict(color=colors[6])  
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            tickvals=[-0.1,-0.75,0,0.1],
            tickangle=0
        )),
    showlegend=True,
    title=f"State 7",
        font=dict(
        family="Helvetica", 
        size=20,
        color="black",
    )
        )
fig.show()

In [15]:

fig = go.Figure()
    
fig.add_trace(go.Scatterpolar(
        r=data['State8'],  
        theta=labels,  
        fill='toself',  
        name='state8',  
        line=dict(color=colors[7])  
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            tickvals=[-0.1,0,0.1],
            tickangle=0
        )),
    showlegend=True,
    title=f"State 8",
        font=dict(
        family="Helvetica", 
        size=20,
        color="black",
    )
        )
fig.show()

In [16]:

fig = go.Figure()
    
fig.add_trace(go.Scatterpolar(
        r=data['State9'],  
        theta=labels,  
        fill='toself',  
        name='state9',  
        line=dict(color=colors[8])  
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            tickvals=[-0.2,-0.1,0,0.1],
            tickangle=0
        )),
    showlegend=True,
    title=f"State 9",
        font=dict(
        family="Helvetica", 
        size=20,
        color="black",
    )
        )
fig.show()

In [17]:
fig = go.Figure()
    
fig.add_trace(go.Scatterpolar(
        r=data['State10'],  
        theta=labels,  
        fill='toself',  
        name='state10',  
        line=dict(color=colors[9])  
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            tickvals=[-0.1,0,0.1],
            tickangle=0
        )),
    showlegend=True,
    title=f"State 10",
        font=dict(
        family="Helvetica", 
        size=20,
        color="black",
    )
        )
fig.show()